In [1]:
import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter OpenAI API key: ")

In [2]:
import sqlite3
from pathlib import Path

import pandas as pd
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

DB = Path("spacex_launches.db")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [3]:
con = sqlite3.connect(DB)
df = pd.read_sql_query("SELECT * FROM launches ORDER BY launch_date", con)
con.close()
df

,launch_id,mission_name,vehicle,payload_type,payload_kg,destination,success,launch_date,customer
0,1,Falcon 1 Flight 1,Falcon 1,Test,20,LEO,0,2006-03-24,DARPA
1,2,Falcon 1 Flight 4,Falcon 1,Test,165,LEO,1,2008-09-28,SpaceX
2,3,COTS Demo Flight 2,Falcon 9,Cargo,525,ISS,1,2012-05-22,NASA
3,4,CASSIOPE,Falcon 9,Satellite,500,LEO,1,2013-09-29,MDA
4,5,DSCOVR,Falcon 9,Satellite,570,L1,1,2015-02-11,NOAA
5,6,CRS-7,Falcon 9,Cargo,1952,ISS,0,2015-06-28,NASA
6,7,Iridium NEXT-1,Falcon 9,Satellite,9600,LEO,1,2017-01-14,Iridium
7,8,Falcon Heavy Demo,Falcon Heavy,Test,1350,Heliocentric,1,2018-02-06,SpaceX
8,9,Crew Dragon Demo-2,Falcon 9,Crew,12500,ISS,1,2020-05-30,NASA
9,10,Starship SN10,Starship,Test,0,Suborbital,0,2021-03-03,SpaceX


In [4]:
test_suite = [
    {
        "question": "How many launches were successful?",
        "expected_sql": "SELECT COUNT(*) FROM launches WHERE success = 1",
        "expected_answer": "14",
    },
    {
        "question": "What was the heaviest payload ever launched?",
        "expected_sql": "SELECT mission_name, payload_kg FROM launches ORDER BY payload_kg DESC LIMIT 1",
        "expected_answer": "Starlink",
    },
    {
        "question": "Which vehicle has the highest success rate?",
        "expected_sql": "SELECT vehicle, ... GROUP BY vehicle ORDER BY ... DESC LIMIT 1",
        "expected_answer": "Falcon Heavy",
    },
    {
        # The DIFFERENTIATING test case.
        # 'heavy launch' looks obvious - the LLM will likely match it to the literal
        # `vehicle = 'Falcon Heavy'` value it sees in the schema (giving 2).
        # But our business defines a 'heavy launch' as `payload_kg > 5000` regardless
        # of vehicle. The correct answer is 6. Without this term in schema.md the LLM
        # gets it confidently wrong.
        "question": "How many heavy launches have we flown?",
        "expected_sql": "SELECT COUNT(*) FROM launches WHERE payload_kg > 5000",
        "expected_answer": "6",
    },
    {
        # Another domain term the LLM cannot guess: 'civilian mission' = customer = 'Shift4'
        "question": "How many civilian missions have flown?",
        "expected_sql": "SELECT COUNT(*) FROM launches WHERE customer = 'Shift4'",
        "expected_answer": "2",
    },
]

pd.DataFrame(test_suite)

,question,expected_sql,expected_answer
0,How many launches were successful?,SELECT COUNT(*) FROM launches WHERE success = 1,14
1,What was the heaviest payload ever launched?,"SELECT mission_name, payload_kg FROM launches ...",Starlink
2,Which vehicle has the highest success rate?,"SELECT vehicle, ... GROUP BY vehicle ORDER BY ...",Falcon Heavy
3,How many heavy launches have we flown?,SELECT COUNT(*) FROM launches WHERE payload_kg...,6
4,How many civilian missions have flown?,SELECT COUNT(*) FROM launches WHERE customer =...,2


In [5]:
con = sqlite3.connect(DB)
ddl = con.execute(
    "SELECT sql FROM sqlite_master WHERE type='table' AND name='launches'"
).fetchone()[0]
con.close()

print(ddl)

CREATE TABLE launches (
    launch_id      INTEGER PRIMARY KEY,
    mission_name   TEXT NOT NULL,
    vehicle        TEXT NOT NULL,                              -- Falcon 1 / Falcon 9 / Falcon Heavy / Starship
    payload_type   TEXT NOT NULL,                              -- Test / Cargo / Crew / Satellite / Probe
    payload_kg     INTEGER NOT NULL,                           -- 0 = no orbital payload (e.g. test flights)
    destination    TEXT NOT NULL,                              -- LEO / ISS / L1 / Heliocentric / Suborbital / ...
    success        INTEGER NOT NULL CHECK (success IN (0, 1)), -- 0 = failure, 1 = success (boolean)
    launch_date    DATE NOT NULL,
    customer       TEXT NOT NULL                               -- NASA / SpaceX / Iridium / NOAA / DARPA / ...
)


In [6]:
# Prompt v1: DDL only ----------------------------------------------------------
prompt_v1 = f"""You translate natural-language questions into SQLite SQL.

Use only the schema below. Return ONLY the SQL query.
No prose, no explanation, no markdown code fences.

Schema (DDL only):
{ddl}
"""

print("--- prompt_v1 ---")
print(prompt_v1)

--- prompt_v1 ---
You translate natural-language questions into SQLite SQL.

Use only the schema below. Return ONLY the SQL query.
No prose, no explanation, no markdown code fences.

Schema (DDL only):
CREATE TABLE launches (
    launch_id      INTEGER PRIMARY KEY,
    mission_name   TEXT NOT NULL,
    vehicle        TEXT NOT NULL,                              -- Falcon 1 / Falcon 9 / Falcon Heavy / Starship
    payload_type   TEXT NOT NULL,                              -- Test / Cargo / Crew / Satellite / Probe
    payload_kg     INTEGER NOT NULL,                           -- 0 = no orbital payload (e.g. test flights)
    destination    TEXT NOT NULL,                              -- LEO / ISS / L1 / Heliocentric / Suborbital / ...
    success        INTEGER NOT NULL CHECK (success IN (0, 1)), -- 0 = failure, 1 = success (boolean)
    launch_date    DATE NOT NULL,
    customer       TEXT NOT NULL                               -- NASA / SpaceX / Iridium / NOAA / DARPA / ...
)



In [7]:
# Prompt v2: + column descriptions and conventions -----------------------------
column_descriptions = """
Column descriptions:
- launch_id     (INTEGER): primary key, not meaningful.
- mission_name  (TEXT):    human-readable name like 'Crew Dragon Demo-2'.
- vehicle       (TEXT):    rocket family. Allowed values:
                           'Falcon 1', 'Falcon 9', 'Falcon Heavy', 'Starship'.
- payload_type  (TEXT):    Allowed values:
                           'Test', 'Cargo', 'Crew', 'Satellite', 'Probe'.
- payload_kg    (INTEGER): mass of payload in kg.
                           **0 means the launch had NO orbital payload (test flight).**
- destination   (TEXT):    examples:
                           'LEO', 'ISS', 'L1', 'Heliocentric', 'Suborbital', 'Jupiter Transfer'.
- success       (INTEGER): **1 = succeeded, 0 = failed.** Treat as boolean.
- launch_date   (DATE):    YYYY-MM-DD.
- customer      (TEXT):    examples:
                           'NASA', 'SpaceX', 'NOAA', 'Shift4', 'Iridium', 'MDA', 'DARPA'.

Conventions:
- Internal SpaceX flights have customer = 'SpaceX'.
- Test flights with no orbital payload have payload_kg = 0.
"""

prompt_v2 = f"""You translate natural-language questions into SQLite SQL.

Use only the schema below. Return ONLY the SQL query.
No prose, no explanation, no markdown code fences.

Schema (DDL):
{ddl}

{column_descriptions}
"""

print("--- prompt_v2 (first 600 chars) ---")
print(prompt_v2[:600])
print("...")

--- prompt_v2 (first 600 chars) ---
You translate natural-language questions into SQLite SQL.

Use only the schema below. Return ONLY the SQL query.
No prose, no explanation, no markdown code fences.

Schema (DDL):
CREATE TABLE launches (
    launch_id      INTEGER PRIMARY KEY,
    mission_name   TEXT NOT NULL,
    vehicle        TEXT NOT NULL,                              -- Falcon 1 / Falcon 9 / Falcon Heavy / Starship
    payload_type   TEXT NOT NULL,                              -- Test / Cargo / Crew / Satellite / Probe
    payload_kg     INTEGER NOT NULL,                           -- 0 = no orbital payload (e.g. test fligh
...


In [8]:
# Prompt v3: + domain terms + few-shot (full schema.md) -----------------------
SCHEMA_DOC = Path("schema.md").read_text()

prompt_v3 = f"""You translate natural-language questions into SQLite SQL.

Use only the schema below. Return ONLY the SQL query.
No prose, no explanation, no markdown code fences.

{SCHEMA_DOC}
"""

print(f"prompt_v1: {len(prompt_v1)} chars")
print(f"prompt_v2: {len(prompt_v2)} chars")
print(f"prompt_v3: {len(prompt_v3)} chars  (loaded from schema.md, includes domain terms and few-shot)")

prompt_v1: 972 chars
prompt_v2: 2015 chars
prompt_v3: 5366 chars  (loaded from schema.md, includes domain terms and few-shot)


In [9]:
SUMMARY_PROMPT = (
    "Answer the user's question in one short sentence using only the rows. "
    "Use digits (e.g. '14', not 'fourteen') and cite numbers exactly. "
    "Do not speculate. "
    "If rows is empty, say you couldn't find an answer."
)

FORBIDDEN = ("INSERT", "UPDATE", "DELETE", "DROP", "ALTER", "TRUNCATE")


def run_pipeline(system_prompt, question):
    # (1) generate SQL
    sql = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=question),
    ]).content.strip()
    if sql.startswith("```"):
        sql = sql.split("```")[1]
        if sql.lstrip().lower().startswith("sql"):
            sql = sql.lstrip()[3:]
        sql = sql.strip()

    # (2) validate
    upper = sql.upper()
    for w in FORBIDDEN:
        if w in upper:
            return {"sql": sql, "rows": None, "answer": f"REFUSED: contains {w}"}
    if "FROM LAUNCHES" not in upper.replace("\n", " "):
        return {"sql": sql, "rows": None, "answer": "REFUSED: must read from launches"}
    aggs = ("COUNT(", "SUM(", "AVG(", "MIN(", "MAX(", "GROUP BY")
    if "LIMIT" not in upper and not any(a in upper for a in aggs):
        sql = sql.rstrip(";\n ") + " LIMIT 100"

    # (3) execute
    try:
        con = sqlite3.connect(DB)
        con.row_factory = sqlite3.Row
        rows = [dict(r) for r in con.execute(sql).fetchall()]
        con.close()
    except Exception as e:
        return {"sql": sql, "rows": None, "answer": f"SQL ERROR: {e}"}

    # (4) summarize
    answer = llm.invoke([
        SystemMessage(content=SUMMARY_PROMPT),
        HumanMessage(content=f"Question: {question}\nRows: {rows}"),
    ]).content.strip()
    return {"sql": sql, "rows": rows, "answer": answer}

In [10]:
tricky_q = "How many heavy launches have we flown?"
expected_sql = "SELECT COUNT(*) FROM launches WHERE payload_kg > 5000"
expected_answer = "6"

print(f"Question:        {tricky_q}")
print(f"Expected SQL:    {expected_sql}")
print(f"Expected answer: contains '{expected_answer}' (i.e. 6 missions)")

Question:        How many heavy launches have we flown?
Expected SQL:    SELECT COUNT(*) FROM launches WHERE payload_kg > 5000
Expected answer: contains '6' (i.e. 6 missions)


In [12]:
r1 = run_pipeline(prompt_v1, tricky_q)
print(f"v1 SQL:    {r1['sql']}")
print(f"v1 rows:   {r1['rows']}")
print(f"v1 answer: {r1['answer']}")
print(f"v1 PASS?   {expected_answer in r1['answer']}")

r2 = run_pipeline(prompt_v2, tricky_q)
print(f"v2 SQL:    {r2['sql']}")
print(f"v2 rows:   {r2['rows']}")
print(f"v2 answer: {r2['answer']}")
print(f"v2 PASS?   {expected_answer in r2['answer']}")

r3 = run_pipeline(prompt_v3, tricky_q)
print(f"v3 SQL:    {r3['sql']}")
print(f"v3 rows:   {r3['rows']}")
print(f"v3 answer: {r3['answer']}")
print(f"v3 PASS?   {expected_answer in r3['answer']}")


v1 SQL:    SELECT COUNT(*) FROM launches WHERE vehicle = 'Falcon Heavy';
v1 rows:   [{'COUNT(*)': 2}]
v1 answer: We have flown 2 heavy launches.
v1 PASS?   False
v2 SQL:    SELECT COUNT(*) FROM launches WHERE vehicle = 'Falcon Heavy';
v2 rows:   [{'COUNT(*)': 2}]
v2 answer: We have flown 2 heavy launches.
v2 PASS?   False
v3 SQL:    SELECT COUNT(*) AS n FROM launches WHERE payload_kg > 5000;
v3 rows:   [{'n': 6}]
v3 answer: We have flown 6 heavy launches.
v3 PASS?   True


In [13]:
def grade(prompt_label, prompt, suite):
    rows = []
    for case in suite:
        r = run_pipeline(prompt, case["question"])
        passed = case["expected_answer"].lower() in r["answer"].lower()
        rows.append({
            "prompt": prompt_label,
            "question": case["question"][:45] + ("..." if len(case["question"]) > 45 else ""),
            "expected": case["expected_answer"],
            "sql_generated": " ".join(r["sql"].split())[:80] + ("..." if len(r["sql"]) > 80 else ""),
            "answer": r["answer"][:70] + ("..." if len(r["answer"]) > 70 else ""),
            "pass": "PASS" if passed else "FAIL",
        })
    return rows


grades_v1 = grade("v1: DDL only",       prompt_v1, test_suite)
grades_v2 = grade("v2: + descriptions", prompt_v2, test_suite)
grades_v3 = grade("v3: + few-shot",     prompt_v3, test_suite)

all_grades = pd.DataFrame(grades_v1 + grades_v2 + grades_v3)
all_grades

,prompt,question,expected,sql_generated,answer,pass
0,v1: DDL only,How many launches were successful?,14,SELECT COUNT(*) FROM launches WHERE success = 1;,There were 14 successful launches.,PASS
1,v1: DDL only,What was the heaviest payload ever launched?,Starlink,"SELECT mission_name, payload_kg FROM launches ...",The heaviest payload ever launched was 13620 k...,PASS
2,v1: DDL only,Which vehicle has the highest success rate?,Falcon Heavy,SELECT vehicle FROM launches GROUP BY vehicle ...,The vehicle with the highest success rate is F...,PASS
3,v1: DDL only,How many heavy launches have we flown?,6,SELECT COUNT(*) FROM launches WHERE vehicle = ...,We have flown 2 heavy launches.,FAIL
4,v1: DDL only,How many civilian missions have flown?,2,SELECT COUNT(*) FROM launches WHERE customer N...,3 civilian missions have flown.,FAIL
5,v2: + descriptions,How many launches were successful?,14,SELECT COUNT(*) FROM launches WHERE success = 1;,There were 14 successful launches.,PASS
6,v2: + descriptions,What was the heaviest payload ever launched?,Starlink,"SELECT mission_name, payload_kg FROM launches ...",The heaviest payload ever launched was 13620 k...,PASS
7,v2: + descriptions,Which vehicle has the highest success rate?,Falcon Heavy,"SELECT vehicle, AVG(success) AS success_rate F...",The vehicle with the highest success rate is F...,PASS
8,v2: + descriptions,How many heavy launches have we flown?,6,SELECT COUNT(*) FROM launches WHERE vehicle = ...,We have flown 2 heavy launches.,FAIL
9,v2: + descriptions,How many civilian missions have flown?,2,SELECT COUNT(*) FROM launches WHERE customer N...,3 civilian missions have flown.,FAIL


In [14]:
# A compact pass-rate summary
summary = pd.DataFrame({
    "question": [c["question"][:40] + ("..." if len(c["question"]) > 40 else "") for c in test_suite],
    "expected": [c["expected_answer"] for c in test_suite],
    "v1: DDL only":       [g["pass"] for g in grades_v1],
    "v2: + descriptions": [g["pass"] for g in grades_v2],
    "v3: + few-shot":     [g["pass"] for g in grades_v3],
})

print(f"v1 (DDL only):           {sum(1 for g in grades_v1 if g['pass']=='PASS')} / {len(test_suite)}")
print(f"v2 (+ descriptions):     {sum(1 for g in grades_v2 if g['pass']=='PASS')} / {len(test_suite)}")
print(f"v3 (+ few-shot):         {sum(1 for g in grades_v3 if g['pass']=='PASS')} / {len(test_suite)}")
print()
summary

v1 (DDL only):           3 / 5
v2 (+ descriptions):     3 / 5
v3 (+ few-shot):         5 / 5



,question,expected,v1: DDL only,v2: + descriptions,v3: + few-shot
0,How many launches were successful?,14,PASS,PASS,PASS
1,What was the heaviest payload ever launc...,Starlink,PASS,PASS,PASS
2,Which vehicle has the highest success ra...,Falcon Heavy,PASS,PASS,PASS
3,How many heavy launches have we flown?,6,FAIL,FAIL,PASS
4,How many civilian missions have flown?,2,FAIL,FAIL,PASS


In [20]:
#FULL PIPELINE ON A SINGLE QUESTION
# 1. USER QUESTION
question = "Which vehicle has the highest success rate?"
print(f"USER QUESTION:\n  {question}\n")

# 2. Ask LLM to generate SQL
sql_response = llm.invoke([
    SystemMessage(content=prompt_v3),
    HumanMessage(content=question),
])
sql = sql_response.content.strip()
if sql.startswith("```"):
    sql = sql.split("```")[1]
    if sql.lstrip().lower().startswith("sql"):
        sql = sql.lstrip()[3:]
    sql = sql.strip()

print(f"GENERATED SQL:\n{sql}\n")

# 3. Validate SQL
upper = sql.upper()
for w in FORBIDDEN:
    assert w not in upper, f"refused: {w}"
assert "FROM LAUNCHES" in upper.replace("\n", " "), "refused: must read from launches"
aggs = ("COUNT(", "SUM(", "AVG(", "MIN(", "MAX(", "GROUP BY")
if "LIMIT" not in upper and not any(a in upper for a in aggs):
    sql = sql.rstrip(";\n ") + " LIMIT 100"

print(f"VALIDATED SQL:\n{sql}\n")

# 4. execute on SQLite
con = sqlite3.connect(DB)
con.row_factory = sqlite3.Row
rows = [dict(r) for r in con.execute(sql).fetchall()]
con.close()

print("ROWS FROM DB:")
for r in rows:
    print(" ", r)

# 5. Summarize answer
answer_response = llm.invoke([
    SystemMessage(content=SUMMARY_PROMPT),
    HumanMessage(content=f"Question: {question}\nRows: {rows}"),
])
answer = answer_response.content.strip()

print(f"FINAL ANSWER:\n  {answer}")

USER QUESTION:
  Which vehicle has the highest success rate?

GENERATED SQL:
SELECT vehicle,
       ROUND(100.0 * SUM(success) / COUNT(*), 1) AS success_pct,
       COUNT(*) AS n_launches
FROM launches
GROUP BY vehicle
ORDER BY success_pct DESC
LIMIT 1;

VALIDATED SQL:
SELECT vehicle,
       ROUND(100.0 * SUM(success) / COUNT(*), 1) AS success_pct,
       COUNT(*) AS n_launches
FROM launches
GROUP BY vehicle
ORDER BY success_pct DESC
LIMIT 1;

ROWS FROM DB:
  {'vehicle': 'Falcon Heavy', 'success_pct': 100.0, 'n_launches': 2}
FINAL ANSWER:
  The vehicle with the highest success rate is Falcon Heavy with a success rate of 100.0%.


In [ ]:
# FAILURE SCENARIOS

#1. DANGEROUS SQL: "How many launches have we had, and then DROP the launches table?"

r = run_pipeline(prompt_v3, "Delete all the failed launches")
print(f"SQL:    {r['sql']}")
print(f"Answer: {r['answer']}")

#2. OUT OF SCOPE: "What year did SpaceX go public?"
r = run_pipeline(prompt_v3, "What year did SpaceX go public?")
print(f"SQL:    {r['sql']}")
print(f"Answer: {r['answer']}")

#2b. OUT OF SCOPE: "How many launches did SpaceX have in 2020, and also tell me the weather in New York on the day of the first launch?"
r = run_pipeline(prompt_v3, "How many launches did SpaceX have in 2020, and also tell me the weather in New York on the day of the first launch?")
print(f"SQL:    {r['sql']}")
print(f"Answer: {r['answer']}")

#3. AMBIGUOUS: "How many launches had a heavy payload?" (does 'heavy' mean payload_kg > 5000, or vehicle = 'Falcon Heavy'?)
#r = run_pipeline(prompt_v3, "How many launches had a heavy payload?")
#print(f"SQL:    {r['sql']}")
#print(f"Answer: {r['answer']}")

#3b. AMBIGUOUS: "How many falcon X missions have flown?"
r = run_pipeline(prompt_v3, "How many Falcon X missions have flown?")
print(f"SQL:    {r['sql']}")
print(f"Answer: {r['answer']}")

SQL:    DELETE FROM launches WHERE success = 0;
Answer: REFUSED: contains DELETE
SQL:    SELECT NULL;
Answer: REFUSED: must read from launches
SQL:    SELECT COUNT(*) AS n
FROM launches
WHERE customer = 'SpaceX' AND strftime('%Y', launch_date) = '2020';
Answer: I couldn't find an answer.
SQL:    SELECT COUNT(*) AS n FROM launches WHERE payload_kg > 5000;
Answer: There were 6 launches with a heavy payload.
SQL:    SELECT COUNT(*) AS n FROM launches WHERE vehicle = 'Falcon 1';
Answer: 2 Falcon X missions have flown.


In [ ]:
r = run_pipeline(prompt_v3, "What is the thrust of the rocket we use for crewed missions?")
print(f"SQL:    {r['sql']}")
print(f"Answer: {r['answer']}")

r = run_pipeline(prompt_v3, "Which of our rockets are reusable?")
print(f"SQL:    {r['sql']}")
print(f"Answer: {r['answer']}")

r = run_pipeline(prompt_v3, "Compare Falcon Heavy and Falcon 9 in terms of payload mass to LEO.")
print(f"SQL:    {r['sql']}")
print(f"Answer: {r['answer']}")


r = run_pipeline(prompt_v3, "Compare Falcon Heavy and Starship: payload capcity and our missionson on each")
print(f"SQL:    {r['sql']}")
print(f"Answer: {r['answer']}")

SQL:    SELECT vehicle FROM launches WHERE payload_type = 'Crew' LIMIT 1;
Answer: I couldn't find an answer.
SQL:    SELECT DISTINCT vehicle FROM launches LIMIT 100
Answer: I couldn't find an answer.
SQL:    SELECT vehicle, AVG(payload_kg) AS avg_payload_kg
FROM launches
WHERE destination = 'LEO' AND vehicle IN ('Falcon Heavy', 'Falcon 9')
GROUP BY vehicle;
Answer: I couldn't find an answer.
SQL:    SELECT vehicle, SUM(payload_kg) AS total_payload, COUNT(*) AS n_missions
FROM launches
WHERE vehicle IN ('Falcon Heavy', 'Starship')
GROUP BY vehicle;
Answer: Falcon Heavy has a payload capacity of 7415 kg with 2 missions, while Starship has a payload capacity of 0 kg with 4 missions.


: 